In [2]:
!pip install netCDF4 scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 69.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.0 MB/s eta 0:00:00


In [3]:
# ============================================================
# CP6 — Lead 05 (n=6)  |  Improved Training
# ============================================================

import os, gc, random, shutil
import numpy as np
import netCDF4 as nc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

BASE     = '/kaggle/input/datasets/divyanshyecho/enso-ham2019-dataset'
CKPT_DIR = '/kaggle/working/cp6_lead05'
os.makedirs(CKPT_DIR, exist_ok=True)

LEAD       = 5
EPOCHS     = 500
SEED       = 42
BATCH_SIZE = 16

CMIP5_INPUT     = f'{BASE}/CMIP5.input.36mn.1861_2001.nc'
SODA_INPUT      = f'{BASE}/SODA.input.36mn.1871_1970.nc'
GODAS_INPUT     = f'{BASE}/GODAS.input.36mn.1980_2015.nc'
CMIP5_LABEL_3MV = f'{BASE}/CMIP5.label.nino34.12mn_3mv.1863_2003.nc'
CMIP5_LABEL_2MV = f'{BASE}/CMIP5.label.nino34.12mn_2mv.1863_2003.nc'
SODA_LABEL_3MV  = f'{BASE}/SODA.label.nino34.12mn_3mv.1873_1972.nc'
SODA_LABEL_2MV  = f'{BASE}/SODA.label.nino34.12mn_2mv.1873_1972.nc'
GODAS_LABEL_3MV = f'{BASE}/GODAS.label.12mn_3mv.1982_2017.nc'
GODAS_LABEL_2MV = f'{BASE}/GODAS.label.12mn_2mv.1982_2017.nc'

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)
print(f'Lead: {LEAD}  n={LEAD+1}  Epochs: {EPOCHS}')

# ── Ocean mask ──────────────────────────────────────────────
ds_soda  = nc.Dataset(SODA_INPUT)
sst_all  = np.array(ds_soda.variables['sst'][:, 0, :, :])
t300_all = np.array(ds_soda.variables['t300'][:, 0, :, :])
lats     = np.array(ds_soda.variables['lat'][:])
lons     = np.array(ds_soda.variables['lon'][:])
ds_soda.close()

land_mask  = (sst_all == 0.0).all(axis=0) & (t300_all == 0.0).all(axis=0)
ocean_mask = ~land_mask
ocean_idx  = np.where(ocean_mask.flatten())[0]
N_NODES    = int(ocean_mask.sum())

lat_grid = np.repeat(lats[:, None], 72, axis=1).flatten()[ocean_idx]
lon_grid = np.repeat(lons[None, :], 24, axis=0).flatten()[ocean_idx]
print(f'Ocean nodes: {N_NODES}')

# ── Static features ─────────────────────────────────────────
ds_s = nc.Dataset(SODA_INPUT)
sr   = np.array(ds_s.variables['sst'][:]).astype(np.float32)
tr   = np.array(ds_s.variables['t300'][:]).astype(np.float32)
ds_s.close()

_X = np.stack([sr, tr], axis=1)
_X = np.nan_to_num(_X, nan=0.0)
_X = _X.reshape(100, 2, 36, -1)[:, :, :, ocean_idx]
_X = _X.transpose(0, 3, 1, 2).reshape(100, N_NODES, -1)

soda_sst_mean  = _X[:, :, :36].mean(axis=(0, 2))
soda_t300_mean = _X[:, :, 36:].mean(axis=(0, 2))
lat_norm = (lat_grid - lat_grid.mean()) / (lat_grid.std() + 1e-6)
lon_norm = (lon_grid - lon_grid.mean()) / (lon_grid.std() + 1e-6)

static_np = np.stack([soda_sst_mean, soda_t300_mean, lat_norm, lon_norm], axis=1)
X_static  = torch.tensor(static_np, dtype=torch.float32).to(device)
del _X, sr, tr; gc.collect()
print(f'Static features: {X_static.shape}')

# ── Data loading ─────────────────────────────────────────────
def load_input(input_file, sst_var='sst'):
    ds   = nc.Dataset(input_file)
    sst  = np.array(ds.variables[sst_var][:]).astype(np.float32)
    t300 = np.array(ds.variables['t300'][:]).astype(np.float32)
    ds.close()
    X = np.stack([sst, t300], axis=1)
    X = np.nan_to_num(X, nan=0.0)
    mean = X.mean(axis=(0, 2), keepdims=True)
    std  = X.std( axis=(0, 2), keepdims=True) + 1e-6
    X    = (X - mean) / std
    N    = X.shape[0]
    X    = X.reshape(N, 2, 36, -1)[:, :, :, ocean_idx]
    X    = X.transpose(0, 3, 1, 2).reshape(N, N_NODES, -1)
    return X

def load_labels(label_file):
    ds = nc.Dataset(label_file)
    pr = np.array(ds.variables['pr'][:]).astype(np.float32)
    ds.close()
    return pr

def get_lead_data(lead):
    if lead == 0:
        cmip5_lbl = CMIP5_LABEL_2MV
        soda_lbl  = SODA_LABEL_2MV
        godas_lbl = GODAS_LABEL_2MV
    else:
        cmip5_lbl = CMIP5_LABEL_3MV
        soda_lbl  = SODA_LABEL_3MV
        godas_lbl = GODAS_LABEL_3MV

    X_cmip5 = load_input(CMIP5_INPUT, sst_var='sst1')
    X_soda  = load_input(SODA_INPUT,  sst_var='sst')
    X_godas = load_input(GODAS_INPUT, sst_var='sst')
    L_cmip5 = load_labels(cmip5_lbl)
    L_soda  = load_labels(soda_lbl)
    L_godas = load_labels(godas_lbl)

    X_cmip5 = X_cmip5[:-1];  Y_cmip5 = L_cmip5[1:, lead, 0, 0]
    X_soda  = X_soda[:-1];   Y_soda  = L_soda[1:,  lead, 0, 0]
    X_godas = X_godas[:-1];  Y_godas = L_godas[1:, lead, 0, 0]

    X_train = np.concatenate([X_cmip5, X_soda], axis=0)
    Y_train = np.concatenate([Y_cmip5, Y_soda], axis=0)
    return X_train, Y_train, X_godas, Y_godas

print('Data loading ready.')

# ── Model ────────────────────────────────────────────────────
class StructureLearner(nn.Module):
    def __init__(self, static_feats=4, hidden=16, a1=0.1, a2=2.0):
        super().__init__()
        self.a1 = a1; self.a2 = a2
        self.lin1 = nn.Linear(static_feats, hidden, bias=False)
        self.lin2 = nn.Linear(static_feats, hidden, bias=False)

    def forward(self, X_static):
        M1 = torch.tanh(self.a1 * self.lin1(X_static))
        M2 = torch.tanh(self.a1 * self.lin2(X_static))
        A  = torch.sigmoid(self.a2 * (M1 @ M2.T))
        A  = A / (A.sum(dim=1, keepdim=True) + 1e-6)
        A  = 0.5 * A + 0.5 * torch.eye(N_NODES, device=A.device)
        return A

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W  = nn.Linear(in_dim, out_dim, bias=False)
        self.bn = nn.BatchNorm1d(out_dim)

    def forward(self, A, X):
        out = torch.stack([A @ X[i] for i in range(X.size(0))], dim=0)
        out = self.W(out)
        B, N, D = out.shape
        out = self.bn(out.reshape(B*N, D)).reshape(B, N, D)
        return F.elu(out)

class GraphinoLSTM(nn.Module):
    def __init__(self, in_vars=2, n_months=36, lstm_hidden=64,
                 gcn_hidden=250, static_feats=4):
        super().__init__()
        self.n_months    = n_months
        self.in_vars     = in_vars
        self.lstm_hidden = lstm_hidden
        self.chunk_size  = 100

        self.lstm = nn.LSTM(input_size=in_vars, hidden_size=lstm_hidden,
                            num_layers=1, batch_first=True)
        self.structure = StructureLearner(static_feats=static_feats)
        self.gcn1      = GCNLayer(lstm_hidden, gcn_hidden)
        self.gcn2      = GCNLayer(gcn_hidden,  gcn_hidden)
        self.mlp = nn.Sequential(
            nn.Linear(2 * gcn_hidden, 128),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, X, X_static):
        B, N, _ = X.shape
        sst  = X[:, :, :self.n_months]
        t300 = X[:, :, self.n_months:]
        seq  = torch.stack([sst, t300], dim=-1)
        node_feats = []
        for start in range(0, N, self.chunk_size):
            end = min(start + self.chunk_size, N)
            s   = seq[:, start:end, :, :].reshape(B * (end - start), self.n_months, self.in_vars)
            _, (h_n, _) = self.lstm(s)
            node_feats.append(h_n.squeeze(0).reshape(B, end - start, self.lstm_hidden))
        node_feat = torch.cat(node_feats, dim=1)
        A  = self.structure(X_static)
        Z1 = self.gcn1(A, node_feat)
        Z2 = self.gcn2(A, Z1) + Z1
        Z  = torch.cat([Z1, Z2], dim=-1)
        g  = Z.mean(dim=1)
        return self.mlp(g).squeeze(-1)

set_seed(SEED)
_m = GraphinoLSTM().to(device)
print(f'Parameters: {sum(p.numel() for p in _m.parameters()):,}')
_x = torch.randn(4, N_NODES, 72).to(device)
with torch.no_grad():
    _o = _m(_x, X_static)
print(f'Output shape: {_o.shape}')
del _m, _x, _o; gc.collect()
print('Architecture OK')

# ── Train ────────────────────────────────────────────────────
def train_lead(lead):
    print(f'\n{"="*55}')
    print(f'Training lead={lead}  n={lead+1}  [CP6 Improved]')
    print(f'{"="*55}')

    pred_file   = f'{CKPT_DIR}/preds_lead{lead:02d}_seed{SEED}.npy'
    latest_path = f'{CKPT_DIR}/latest_lead{lead:02d}_seed{SEED}.pt'
    best_path   = f'{CKPT_DIR}/best_lead{lead:02d}_seed{SEED}.pt'

    if os.path.exists(pred_file):
        print('  Already done — loading saved predictions.')
        _, _, _, Y_test = get_lead_data(lead)
        preds = np.load(pred_file)
        cc    = pearsonr(preds, Y_test)[0]
        print(f'  CC = {cc:.4f}')
        return preds, Y_test

    X_train, Y_train, X_test, Y_test = get_lead_data(lead)
    X_tr = torch.tensor(X_train, dtype=torch.float32)
    Y_tr = torch.tensor(Y_train, dtype=torch.float32)
    X_te = torch.tensor(X_test,  dtype=torch.float32)

    loader = DataLoader(TensorDataset(X_tr, Y_tr),
                        batch_size=BATCH_SIZE, shuffle=True)

    set_seed(SEED)
    model = GraphinoLSTM().to(device)
    opt   = torch.optim.SGD(model.parameters(), lr=0.005,
                            momentum=0.9, nesterov=True, weight_decay=1e-6)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
                opt, T_max=EPOCHS, eta_min=1e-5)

    best_cc = -999.0; start_epoch = 1

    if os.path.exists(latest_path):
        print('  Resuming from checkpoint...')
        ckpt = torch.load(latest_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        opt.load_state_dict(ckpt['opt_state'])
        sched.load_state_dict(ckpt['sched_state'])
        best_cc     = ckpt['best_cc']
        start_epoch = ckpt['epoch'] + 1
        print(f'  Resumed at epoch {start_epoch} | best CC = {best_cc:.4f}')
    else:
        print('  Starting fresh.')

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb, X_static)
            loss = F.mse_loss(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(loader)
        sched.step()

        model.eval()
        preds_te = []
        with torch.no_grad():
            for i in range(0, len(X_te), BATCH_SIZE):
                xb = X_te[i:i+BATCH_SIZE].to(device)
                preds_te.append(model(xb, X_static).cpu().numpy())
        preds_te = np.concatenate(preds_te)
        cc = pearsonr(preds_te, Y_test)[0]

        if cc > best_cc:
            best_cc = cc
            torch.save(model.state_dict(), best_path)

        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'opt_state': opt.state_dict(), 'sched_state': sched.state_dict(),
            'best_cc': best_cc,
        }, latest_path)

        if epoch % 25 == 0 or epoch == 1:
            lr = opt.param_groups[0]['lr']
            print(f'  Epoch {epoch:3d} | loss={avg_loss:.4f} | '
                  f'CC={cc:.4f} | best={best_cc:.4f} | lr={lr:.6f}')

    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=False))
    model.eval()
    preds_best = []
    with torch.no_grad():
        for i in range(0, len(X_te), BATCH_SIZE):
            xb = X_te[i:i+BATCH_SIZE].to(device)
            preds_best.append(model(xb, X_static).cpu().numpy())
    preds_best = np.concatenate(preds_best)

    np.save(pred_file, preds_best)
    if os.path.exists(latest_path):
        os.remove(latest_path)

    print(f'\n  DONE. Best CC = {best_cc:.4f}')
    del model, X_tr, Y_tr, X_te, X_train, Y_train, X_test
    gc.collect(); torch.cuda.empty_cache()
    return preds_best, Y_test

# ── Run ──────────────────────────────────────────────────────
preds, labels = train_lead(LEAD)
cc = pearsonr(preds, labels)[0]
print(f'\nLead {LEAD:02d} (n={LEAD+1}) Final CC = {cc:.4f}')

# ── Save for download ────────────────────────────────────────
src = f'{CKPT_DIR}/preds_lead{LEAD:02d}_seed{SEED}.npy'
dst = f'/kaggle/working/preds_lead{LEAD:02d}_seed{SEED}.npy'
shutil.copy(src, dst)
print(f'Saved: preds_lead{LEAD:02d}_seed{SEED}.npy')
print('Now click Save & Run All to persist.')

Device: cuda
Lead: 5  n=6  Epochs: 500
Ocean nodes: 1393
Static features: torch.Size([1393, 4])
Data loading ready.
Parameters: 161,549
Output shape: torch.Size([4])
Architecture OK

Training lead=5  n=6  [CP6 Improved]
  Starting fresh.
  Epoch   1 | loss=0.5242 | CC=-0.0731 | best=-0.0731 | lr=0.005000
  Epoch  25 | loss=0.4716 | CC=-0.0707 | best=0.0608 | lr=0.004969
  Epoch  50 | loss=0.4652 | CC=0.0658 | best=0.1154 | lr=0.004878
  Epoch  75 | loss=0.4698 | CC=0.1572 | best=0.1720 | lr=0.004728
  Epoch 100 | loss=0.4639 | CC=0.1412 | best=0.2503 | lr=0.004523
  Epoch 125 | loss=0.4408 | CC=0.1814 | best=0.2814 | lr=0.004269
  Epoch 150 | loss=0.4403 | CC=0.3214 | best=0.3225 | lr=0.003972
  Epoch 175 | loss=0.4191 | CC=0.2799 | best=0.3586 | lr=0.003638
  Epoch 200 | loss=0.4143 | CC=0.2111 | best=0.3586 | lr=0.003276
  Epoch 225 | loss=0.4069 | CC=0.2697 | best=0.3944 | lr=0.002895
  Epoch 250 | loss=0.3959 | CC=0.3210 | best=0.4250 | lr=0.002505
  Epoch 275 | loss=0.3890 | CC=0.